# 05. 종합 실험 — 실제 포트폴리오 데이터 적용

## 실험 목표
- 01~04에서 구축한 파이프라인을 **실제 포트폴리오 데이터**에 적용
- 에이전트 응답 품질 체계적 평가 (정확성, 유용성, 누락 여부)
- 한계점 및 개선 방향 도출

## 실험 대상 데이터
1. `sample_data.csv` — 기본 검증용 (타이타닉 유사 데이터)
2. `01_Daily_Project` 프로젝트 결과물 (ETF/금융 데이터)
3. (선택) 직접 업로드한 CSV 파일

---
## 0. 전체 파이프라인 모듈 로드

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.gemini import GeminiModel

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
print("✅ 환경 설정 완료")

In [ ]:
# ─────────────────────────────────────────────
# 스키마 정의 (04 노트북에서 가져옴)
# ─────────────────────────────────────────────
class DataOverview(BaseModel):
    rows: int
    columns: int
    numeric_columns: list[str]
    categorical_columns: list[str]
    total_missing_cells: int

class MissingInfo(BaseModel):
    column_name: str
    missing_count: int
    missing_ratio: float
    recommendation: str

class OutlierInfo(BaseModel):
    column_name: str
    outlier_count: int
    outlier_ratio: float
    lower_bound: float
    upper_bound: float
    recommendation: str

class CorrelationHighlight(BaseModel):
    col_a: str
    col_b: str
    correlation: float
    interpretation: str

class KeyInsight(BaseModel):
    category: str
    insight: str
    priority: str

class EDAReport(BaseModel):
    dataset_name: str
    overview: DataOverview
    missing_analysis: list[MissingInfo]
    outlier_analysis: list[OutlierInfo]
    correlation_highlights: list[CorrelationHighlight]
    key_insights: list[KeyInsight]
    next_steps: list[str]

print("✅ 스키마 로드 완료")

In [ ]:
# ─────────────────────────────────────────────
# Tool 함수 정의
# ─────────────────────────────────────────────
def describe_data(df):
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {"shape": {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
            "numeric_columns": numeric_cols, "categorical_columns": cat_cols,
            "numeric_summary": df[numeric_cols].describe().round(2).to_dict()}

def check_missing(df):
    mc = df.isnull().sum()
    mr = (mc / len(df) * 100).round(2)
    res = pd.DataFrame({'count': mc, 'ratio(%)': mr}).query('count > 0').sort_values('count', ascending=False)
    return {"total_missing_cells": int(df.isnull().sum().sum()),
            "columns_with_missing": res.to_dict(orient='index')}

def detect_outliers_all(df):
    results = {}
    for col in df.select_dtypes(include='number').columns:
        s = df[col].dropna()
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
        out = df[(df[col] < lo) | (df[col] > hi)][col]
        results[col] = {"outlier_count": len(out),
                        "outlier_ratio(%)": round(len(out)/len(s)*100, 2),
                        "lower_bound": round(float(lo), 3),
                        "upper_bound": round(float(hi), 3)}
    return results

def correlation_top(df, method='pearson', top_n=5):
    num = df.select_dtypes(include='number')
    corr = num.corr(method=method).round(3)
    pairs = [(corr.columns[i], corr.columns[j], float(corr.iloc[i,j]))
             for i in range(len(corr.columns)) for j in range(i+1, len(corr.columns))]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    return [{"col_a": a, "col_b": b, "corr": v} for a, b, v in pairs[:top_n]]

def visualize_summary(df):
    """주요 수치형 컬럼 분포 요약 시각화"""
    num_cols = df.select_dtypes(include='number').columns.tolist()
    n = min(len(num_cols), 6)
    if n == 0:
        return {"message": "수치형 컬럼 없음"}
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 7)) if n > 3 else plt.subplots(1, n, figsize=(5*n, 4))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    for i, col in enumerate(num_cols[:n]):
        s = df[col].dropna()
        s.plot(kind='hist', bins=20, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].set_title(col, fontsize=10)
        axes[i].axvline(s.mean(), color='red', linestyle='--', linewidth=1)
    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('수치형 컬럼 분포 요약', fontsize=13)
    plt.tight_layout()
    plt.show()
    return {"visualized_columns": num_cols[:n]}

print("✅ Tool 함수 정의 완료")

In [ ]:
# ─────────────────────────────────────────────
# 에이전트 빌더 함수 — 데이터셋별로 재사용
# ─────────────────────────────────────────────
@dataclass
class DataDeps:
    df: pd.DataFrame
    dataset_name: str

def build_eda_agent(api_key: str) -> Agent:
    model = GeminiModel(model_name="gemini-2.0-flash", api_key=api_key)
    agent = Agent(
        model=model,
        deps_type=DataDeps,
        result_type=EDAReport,
        system_prompt="""
        EDA 자동화 에이전트입니다. 모든 Tool을 반드시 실행한 후
        분석 결과를 EDAReport 스키마에 정확히 맞춰 반환하십시오.
        수치는 반드시 Tool 실행 결과를 그대로 사용하십시오.
        """
    )
    
    @agent.tool
    def tool_describe_data(ctx: RunContext[DataDeps]) -> dict:
        """데이터 기본 통계량을 반환합니다."""
        return describe_data(ctx.deps.df)
    
    @agent.tool
    def tool_check_missing(ctx: RunContext[DataDeps]) -> dict:
        """결측치 현황을 분석합니다."""
        return check_missing(ctx.deps.df)
    
    @agent.tool
    def tool_detect_outliers_all(ctx: RunContext[DataDeps]) -> dict:
        """모든 수치형 컬럼의 이상치를 일괄 탐지합니다."""
        return detect_outliers_all(ctx.deps.df)
    
    @agent.tool
    def tool_correlation_top(ctx: RunContext[DataDeps], method: str = 'pearson') -> list:
        """상위 상관관계 쌍을 반환합니다."""
        return correlation_top(ctx.deps.df, method)
    
    @agent.tool
    def tool_visualize_summary(ctx: RunContext[DataDeps]) -> dict:
        """주요 수치형 컬럼의 분포를 한 번에 시각화합니다."""
        return visualize_summary(ctx.deps.df)
    
    return agent

print("✅ 에이전트 빌더 준비 완료")

---
## 1. 실험 1: sample_data.csv (기준 검증)

In [ ]:
async def run_full_pipeline(csv_path: str, dataset_name: str) -> tuple[EDAReport, float]:
    """CSV 경로를 받아 EDA 리포트를 생성하고 (리포트, 소요시간)을 반환합니다."""
    df = pd.read_csv(csv_path)
    deps = DataDeps(df=df, dataset_name=dataset_name)
    agent = build_eda_agent(api_key)
    
    print(f"📂 {dataset_name} ({df.shape[0]}행 × {df.shape[1]}열)")
    print("🤖 EDA 에이전트 실행 중...")
    
    start = time.time()
    result = await agent.run(
        f"'{dataset_name}' 데이터에 대해 완전한 EDA 리포트를 생성해주세요.",
        deps=deps
    )
    elapsed = time.time() - start
    
    print(f"✅ 완료 (소요: {elapsed:.1f}초)")
    return result.output, elapsed

report1, t1 = await run_full_pipeline("data/sample_data.csv", "타이타닉 승객 데이터")

In [ ]:
# 시각화 별도 실행
df1 = pd.read_csv("data/sample_data.csv")
visualize_summary(df1)

# 리포트 출력 함수
def print_report(report: EDAReport, elapsed: float):
    print("=" * 60)
    print(f"📋 EDA 리포트: {report.dataset_name}  [{elapsed:.1f}초]")
    print("=" * 60)
    print(f"  크기: {report.overview.rows:,}행 × {report.overview.columns}열")
    print(f"  결측 셀: {report.overview.total_missing_cells:,}개")
    print()
    print("▶ 결측치")
    for m in report.missing_analysis:
        print(f"  • {m.column_name}: {m.missing_count}개({m.missing_ratio:.1f}%) → {m.recommendation}")
    print()
    print("▶ 이상치")
    for o in report.outlier_analysis:
        if o.outlier_count > 0:
            print(f"  • {o.column_name}: {o.outlier_count}개({o.outlier_ratio:.1f}%) → {o.recommendation}")
    print()
    print("▶ 주요 상관관계")
    for c in report.correlation_highlights:
        print(f"  • {c.col_a} ↔ {c.col_b}: {c.correlation:.3f} ({c.interpretation})")
    print()
    print("▶ 핵심 인사이트")
    for i, ins in enumerate(report.key_insights, 1):
        print(f"  {i}. [{ins.priority}] {ins.insight}")
    print()
    print("▶ 권장 후속 작업")
    for i, s in enumerate(report.next_steps, 1):
        print(f"  {i}. {s}")

print_report(report1, t1)

---
## 2. 실험 2: 사용자 정의 CSV 파일

In [ ]:
# ──────────────────────────────────────────────
# 원하는 CSV 경로로 변경하여 실험하십시오
# ──────────────────────────────────────────────
CUSTOM_CSV = r"..\..\01_Daily_Project\PJ05_ETF_Portfolio\data\etf_data.csv"  # 경로 수정
CUSTOM_NAME = "ETF 포트폴리오 데이터"

if os.path.exists(CUSTOM_CSV):
    report2, t2 = await run_full_pipeline(CUSTOM_CSV, CUSTOM_NAME)
    print_report(report2, t2)
    
    # JSON 저장
    with open("data/eda_report_custom.json", 'w', encoding='utf-8') as f:
        json.dump(report2.model_dump(), f, ensure_ascii=False, indent=2)
    print("\n✅ 리포트 저장: data/eda_report_custom.json")
else:
    print(f"⚠️ 파일을 찾을 수 없습니다: {CUSTOM_CSV}")
    print("CUSTOM_CSV 경로를 실제 파일 경로로 수정하십시오.")

---
## 3. 에이전트 응답 품질 평가

In [ ]:
# 직접 계산한 값과 에이전트 출력 비교
df_check = pd.read_csv("data/sample_data.csv")

# 실측값 계산
actual_missing = df_check.isnull().sum().sum()
actual_rows = len(df_check)
actual_num_cols = df_check.select_dtypes(include='number').shape[1]

# 에이전트 출력값
pred_missing = report1.overview.total_missing_cells
pred_rows = report1.overview.rows
pred_num_cols = len(report1.overview.numeric_columns)

print("[에이전트 정확도 검증]")
print(f"{'항목':<20} {'실측값':>10} {'에이전트':>10} {'일치':>8}")
print("-" * 50)
print(f"{'총 결측 셀 수':<20} {actual_missing:>10} {pred_missing:>10} {'✅' if actual_missing==pred_missing else '❌':>8}")
print(f"{'행 수':<20} {actual_rows:>10} {pred_rows:>10} {'✅' if actual_rows==pred_rows else '❌':>8}")
print(f"{'수치형 컬럼 수':<20} {actual_num_cols:>10} {pred_num_cols:>10} {'✅' if actual_num_cols==pred_num_cols else '❌':>8}")

---
## 4. 한계점 및 개선 방향

### 실험을 통해 발견한 한계점

| 한계점 | 원인 | 개선 방향 |
|---|---|---|
| 대용량 데이터 처리 속도 | LLM API 호출 오버헤드 | 비동기 병렬 처리 / 샘플링 |
| 복잡한 도메인 인사이트 부족 | 범용 모델의 도메인 지식 한계 | Few-shot 예시 추가 / 파인튜닝 |
| 시각화 자동 저장 미지원 | matplotlib 렌더링 → LLM 불가 | Base64 인코딩 또는 별도 저장 로직 |
| Tool 호출 비용 | API 토큰 과금 | 캐싱 레이어 추가 |

### 다음 단계 아이디어
1. **모듈화**: Tool 함수를 `da_tools.py`로 분리하여 재사용성 향상
2. **비동기 병렬화**: 여러 컬럼 분석을 동시에 실행
3. **Streamlit 연동**: 웹 UI에서 CSV 업로드 → 자동 EDA 리포트 출력
4. **MCP 연동**: 03_Coding_Study/09_LLM의 MCP 패턴 적용하여 파일시스템 직접 접근

---
## 5. 전체 실험 요약

In [ ]:
print("=" * 60)
print("📚 DA 자동화 에이전트 실험 시리즈 요약")
print("=" * 60)

summary = [
    ("01", "에이전트 기본구성", "PydanticAI + Gemini 연결, 멀티턴 대화"),
    ("02", "데이터분석 도구정의", "5종 Tool 함수 구현 + 에이전트 등록"),
    ("03", "EDA 자동화 파이프라인", "자율 Tool 선택 + 호출 로그 추적"),
    ("04", "구조화 리포트 생성", "Pydantic 스키마 기반 JSON 리포트 자동 생성"),
    ("05", "종합 실험", "실제 데이터 적용 + 품질 평가 + 한계점 분석"),
]

for no, title, desc in summary:
    print(f"  {no}. {title:20} — {desc}")

print()
print("🎯 핵심 성과")
print("  • CSV 한 줄 입력으로 자동 EDA 완료")
print("  • 구조화된 JSON 리포트 자동 생성")
print("  • 재사용 가능한 에이전트 빌더 패턴 확립")
print()
print("📂 생성 파일 목록")
import glob
for f in sorted(glob.glob('**/*', recursive=True)):
    if not f.startswith('.') and '__pycache__' not in f:
        print(f"  {f}")